# Group Lab 1: Scientific Visualization for Earth and Environmental Data

        **Week:** Week 7

        **Lab type:** Group lab

        **Estimated time:** 2 lab periods

        ## Learning objectives

        - Assign group roles.
- Design a readable figure.
- Use labels, units, and legends.
- Write a figure caption.

        ## Earth and environmental motivation

        A good scientific figure makes a data pattern visible and supports a careful interpretation.

        ## Dataset

        Weather and streamflow processed CSV files

        ## Python concepts used

        - Figure design
- Axes
- Legends
- Captions
- Interpretation

## Lab 6 Debrief and Collaborative Debugging (First 20 Minutes)

Open the debrief card from Lab 6. Two to four students or groups will
share a solved problem, an unresolved problem with evidence, or a verification
choice. The class will investigate one open problem one check at a time.

- 0-3 min: review the issue board.
- 3-11 min: student reports.
- 11-18 min: collaborative debugging.
- 18-20 min: record one reusable lesson and connect it to today's Lab.


## Required imports and project paths

Run this cell first. It finds the project root whether the notebook is opened from the
repository root or from a notebook folder.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data folder exists: {PROCESSED_DIR.exists()}")

## Group roles

- Module A: data cleaning and validation.
- Module B: summary statistics.
- Module C: figure design, caption, and interpretation.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

weather = pd.read_csv(PROCESSED_DIR / "iowa_city_weather_daily.csv", parse_dates=["date"])
stream = pd.read_csv(PROCESSED_DIR / "iowa_streamflow_daily.csv", parse_dates=["date"])
recent = stream[stream["date"].dt.year >= 2020]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(recent["date"], recent["discharge_cfs"], color="royalblue", linewidth=0.8, label="Daily discharge")
ax.set_title("Iowa River daily mean discharge")
ax.set_xlabel("Date")
ax.set_ylabel("Discharge (cfs)")
ax.legend()
fig.tight_layout()
plt.show()

## Guided coding: a two-panel story (rain above, river below)

Hydrologists often draw precipitation upside down above the hydrograph so the
rain appears to "fall" onto the river response. `sharex=True` keeps both
panels aligned, and `axvspan` highlights the flood window around the annual
peak. Modules B and C build and refine this figure together.

In [ ]:
year = 2024
weather_year = weather[weather["date"].dt.year == year]
stream_year = stream[stream["date"].dt.year == year]

fig, (ax_rain, ax_flow) = plt.subplots(
    2, 1, figsize=(10, 6), sharex=True, gridspec_kw={"height_ratios": [1, 2]}
)
ax_rain.bar(weather_year["date"], weather_year["precipitation_mm"], width=1.0, color="steelblue")
ax_rain.set_ylabel("Precipitation\n(mm/day)")
ax_rain.set_title(f"Rain and river response, Iowa City, {year}")
ax_rain.invert_yaxis()

ax_flow.plot(stream_year["date"], stream_year["discharge_cfs"], color="navy", linewidth=1.0)
ax_flow.set_ylabel("Discharge (cfs)")
ax_flow.set_xlabel("Date")

peak_date = stream_year.loc[stream_year["discharge_cfs"].idxmax(), "date"]
for panel in (ax_rain, ax_flow):
    panel.axvspan(peak_date - pd.Timedelta(days=7), peak_date + pd.Timedelta(days=7),
                  color="orange", alpha=0.2)
fig.tight_layout()
plt.show()
print(f"Highlighted window is centered on the {year} peak flow: {peak_date.date()}")

## Guided coding: the flow-duration curve on a log axis

A log y-axis is the standard choice when values span orders of magnitude.
Q10 (flow exceeded only 10 percent of days) describes floods; Q90 (exceeded
90 percent of days) describes drought and water supply.

In [ ]:
from sees4100.hydro import flow_duration_curve

fdc = flow_duration_curve(stream)
q10 = stream["discharge_cfs"].quantile(0.90)
q90 = stream["discharge_cfs"].quantile(0.10)

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(fdc["exceedance_probability"] * 100, fdc["discharge_cfs"], color="darkslategray")
ax.axhline(q10, color="royalblue", linestyle="--", linewidth=0.9, label=f"Q10 = {q10:,.0f} cfs")
ax.axhline(q90, color="firebrick", linestyle="--", linewidth=0.9, label=f"Q90 = {q90:,.0f} cfs")
ax.set_xlabel("Percent of days flow is exceeded")
ax.set_ylabel("Discharge (cfs)")
ax.set_title("Flow-duration curve, Iowa River at Iowa City")
ax.legend()
ax.grid(True, alpha=0.3, which="both")
fig.tight_layout()
plt.show()

## Guided coding: fix this bad figure (Module C)

The next cell produces a figure with at least five design problems. Run it,
list the problems as a group, then rebuild it properly. Check against this
list: axis labels with units, readable font and figure size, one quantity per
axis (or a labeled twin axis), a legend when two lines share a panel, a title
or caption that states the message, and colors that remain distinct in
grayscale.

In [ ]:
fig, ax = plt.subplots(figsize=(4, 2.2))
ax.plot(stream["date"], stream["discharge_cfs"])
ax.plot(weather["date"], weather["precipitation_mm"])
plt.show()

## Writing the caption (Module C)

A strong caption has three parts: what is shown, how it was produced, and
what the reader should notice. Example skeleton: "Daily discharge of the
Iowa River at Iowa City for 2024 (blue line) with daily precipitation
(inverted bars). Data from USGS and the Iowa Environmental Mesonet. The
highlighted window marks the spring flood, which peaked N days after the
heaviest rainfall." Your 200-word interpretation then expands the third part:
pattern, mechanism, and one caveat about the data or method.

## Try it yourself

Improve the figure by adding an annual mean reference line or by focusing on one hydrologically interesting year.

## Graded Checkpoint: Independent Analysis

The guided cells are examples. Complete the task below with your own code; an
unchanged guided notebook does not meet the submission requirement.

Choose one complete year and create an original two-panel figure that connects precipitation and discharge. The figure must use readable labels, units, a shared time axis, and at least one design choice that remains clear in grayscale. Write a caption that states the scientific message.


In [ ]:
# GRADED CHECKPOINT
# Write your code below. Include at least one verification check.


### Scientific Explanation

Replace this text with your interpretation. State what the result means, cite
one piece of numerical or graphical evidence, and name one limitation.


## More practice

1. Rebuild the two-panel figure for a different year and state in one
   sentence how the flood timing differs from 2024.
2. Use `ax.annotate` to point an arrow at the peak-flow day with a short
   text label.
3. Save your final polished figure with `save_figure` from
   `sees4100.plotting` into a `figures/` folder at the project root, and
   reference the saved file name in your caption.

## Common mistakes and debugging tips

- Check that `PROCESSED_DIR.exists()` printed `True`.
- Read error messages from the bottom upward.
- Check column names with `df.columns` before selecting a column.
- Keep units in figure labels and written interpretations.
- Re-run earlier cells after changing data-loading or helper-code cells.

## Deliverables checklist

        - [ ] One polished figure
- [ ] Caption
- [ ] About 200 words of scientific interpretation
- [ ] Individual contribution statement

        ## Short reflection

        Which figure design choices helped the group communicate the main pattern?

        ## Rubric summary

        Correctness and completion, readable code, labeled figures, interpretation,
        and reproducibility all matter. Your submitted notebook should run from top to bottom.

## Debrief Card for the Next Lab

Complete this before the next Wednesday meeting. An unresolved problem is a
valid and useful report.

**Goal:** Replace this text with what you were trying to calculate or show.

**Expected result:** Replace this text.

**What happened:** Replace this text with the result, error, or design choice.

**Evidence:** Include an error message, value, figure observation, or tiny test.

**What I tried:** Replace this text.

**Fix or next check:** State what fixed it, or what the class should test next.

**Lesson from a classmate:** Complete this during the next debrief.
